# Base RNN Notebook v2 - Pipeline de Retrieval Melhorado

Este notebook implementa uma versao mais robusta do pipeline:
- ingestao idempotente no Chroma (sem duplicar chunks em reexecucoes)
- metadados enriquecidos (pagina, clausula, fonte)
- retrieval com MMR
- avaliacao automatica com base no arquivo gold_standart.md

In [1]:
%pip install -qU langchain-community pypdf langchain-huggingface sentence-transformers langchain-chroma pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import re
import hashlib
import unicodedata

import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

PROJECT_ROOT = Path.cwd().resolve().parents[1]
PDF_PATH = PROJECT_ROOT / "data" / "pablo" / "sindilojas_2025_2026.pdf"
GOLD_PATH = PROJECT_ROOT / "data" / "pablo" / "gold_standart.md"
PERSIST_DIR = Path.cwd() / "chroma_langchain_db"

COLLECTION_NAME = "cct_v2_chunk800_o100_mmr"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
TOP_K = 8
FETCH_K = 30

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PDF_PATH exists: {PDF_PATH.exists()}")
print(f"GOLD_PATH exists: {GOLD_PATH.exists()}")

c:\Users\Técnico em IA\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: D:\UC15\senac_ia_uc15_nlp_project
PDF_PATH exists: True
GOLD_PATH exists: True


In [3]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def detect_clause(text: str) -> str:
    match = re.search(r"(?i)(clausula\s+\d+[a-z]?)", text)
    return match.group(1).lower() if match else "clausula_nao_identificada"


def deterministic_chunk_id(source: str, page: int, clause: str, chunk_text: str) -> str:
    payload = f"{source}|{page}|{clause}|{normalize_text(chunk_text)}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

In [4]:
loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()

for doc in docs:
    page_num = doc.metadata.get("page", -1)
    doc.metadata["source"] = str(PDF_PATH.name)
    doc.metadata["page"] = int(page_num)
    doc.metadata["clause"] = detect_clause(doc.page_content)

print(f"Paginas carregadas: {len(docs)}")
print("Exemplo de metadata:", docs[0].metadata)

Paginas carregadas: 38
Exemplo de metadata: {'producer': 'Microsoft® Word para Microsoft 365 - by Lacuna Software PKI SDK', 'creator': 'Microsoft® Word para Microsoft 365', 'creationdate': '2025-11-24T18:02:52-03:00', 'documentkey': '5ccd231f-5127-4696-84df-b576074c6eba', 'author': 'Vivianne Morais', 'moddate': '2025-11-24T21:54:55+00:00', 'source': 'sindilojas_2025_2026.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1', 'clause': 'clausula_nao_identificada'}


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
)

all_splits = text_splitter.split_documents(docs)
chunk_ids = []

for chunk in all_splits:
    source = chunk.metadata.get("source", "unknown")
    page = int(chunk.metadata.get("page", -1))
    clause = chunk.metadata.get("clause", "clausula_nao_identificada")
    chunk_id = deterministic_chunk_id(source, page, clause, chunk.page_content)
    chunk.metadata["chunk_id"] = chunk_id
    chunk_ids.append(chunk_id)

print(f"Chunks gerados: {len(all_splits)}")
print(f"IDs unicos: {len(set(chunk_ids))}")

Chunks gerados: 178
IDs unicos: 178


In [6]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)

# add_documents com IDs deterministas evita duplicacao em reexecucoes
vector_store.add_documents(documents=all_splits, ids=chunk_ids)

collection_size = vector_store._collection.count()
print(f"Colecao: {COLLECTION_NAME}")
print(f"Total de registros no Chroma: {collection_size}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11996.13it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Colecao: cct_v2_chunk800_o100_mmr
Total de registros no Chroma: 178


In [7]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": TOP_K, "fetch_k": FETCH_K, "lambda_mult": 0.5},
)

question = "Quais sao as condicoes de remuneracao e jornada para o trabalho em feriados, conforme a clausula 41?"
retrieved_docs = retriever.invoke(question)

print(f"Documentos recuperados: {len(retrieved_docs)}\n")
for i, d in enumerate(retrieved_docs[:3], start=1):
    print(f"--- Top {i} ---")
    print("Metadata:", d.metadata)
    print(d.page_content[:400].replace("\n", " ") + "...\n")

Documentos recuperados: 8

--- Top 1 ---
Metadata: {'creationdate': '2025-11-24T18:02:52-03:00', 'page': 18, 'page_label': '19', 'total_pages': 38, 'producer': 'Microsoft® Word para Microsoft 365 - by Lacuna Software PKI SDK', 'clause': 'clausula_nao_identificada', 'start_index': 2057, 'chunk_id': '07c060765a30ce3dec9d536d73248630e3a41bcc5c2b88fab1ceb2eec5429d8e', 'author': 'Vivianne Morais', 'creator': 'Microsoft® Word para Microsoft 365', 'documentkey': '5ccd231f-5127-4696-84df-b576074c6eba', 'moddate': '2025-11-24T21:54:55+00:00', 'source': 'sindilojas_2025_2026.pdf'}
no mesmo e declaração de que está sendo cumprida integralmente a Convenção Coletiva de  Trabalho, sendo este documento o indispensável comprovante da regularidade do trabalho;    b) manifestação de vontade por escrito, por parte do empregado, as sistido o menor por seu  representante legal, em instrumento individual ou plúrimo, do qual conste:    I   – os feriados a serem trabalhados; e  II  – a d...

--- Top 2 ---
Met

In [8]:
def parse_gold_qa(md_path: Path):
    text = md_path.read_text(encoding="utf-8")
    pattern = re.compile(
        r"Pergunta:\s*(.*?)\s*Resposta:\s*(.*?)(?=\n\nQuestao|\Z)",
        re.DOTALL | re.IGNORECASE,
    )
    pairs = []
    for q, a in pattern.findall(text):
        pairs.append({"question": q.strip(), "expected_answer": a.strip()})
    return pairs


gold_pairs = parse_gold_qa(GOLD_PATH)
print(f"Total de pares no gold: {len(gold_pairs)}")
pd.DataFrame(gold_pairs).head(3)

Total de pares no gold: 1


,question,expected_answer
0,Qual é o índice de reajuste salarial estabelec...,"O índice de reajuste é de 6,00% (seis por cent..."


In [9]:
def evaluate_retrieval(gold_pairs, retriever, top_k=TOP_K):
    rows = []

    for i, item in enumerate(gold_pairs, start=1):
        q = item["question"]
        expected = normalize_text(item["expected_answer"])

        docs = retriever.invoke(q)[:top_k]
        ranks = []

        for rank, doc in enumerate(docs, start=1):
            content = normalize_text(doc.page_content)
            if expected in content:
                ranks.append(rank)

        hit = int(len(ranks) > 0)
        rr = 1 / ranks[0] if ranks else 0

        rows.append(
            {
                "idx": i,
                "question": q,
                "hit@k": hit,
                "mrr_contrib": rr,
                "first_relevant_rank": ranks[0] if ranks else None,
            }
        )

    df = pd.DataFrame(rows)
    recall_at_k = df["hit@k"].mean()
    mrr = df["mrr_contrib"].mean()
    return df, recall_at_k, mrr


eval_df, recall_at_k, mrr = evaluate_retrieval(gold_pairs, retriever, top_k=TOP_K)

print(f"Recall@{TOP_K}: {recall_at_k:.2%}")
print(f"MRR@{TOP_K}: {mrr:.4f}")
eval_df

Recall@8: 0.00%
MRR@8: 0.0000


,idx,question,hit@k,mrr_contrib,first_relevant_rank
0,1,Qual é o índice de reajuste salarial estabelec...,0,0,None


## Proximos testes recomendados

1. Testar outros embeddings (ex.: bge-m3 e e5-multilingual).
2. Rodar grid de chunk_size/chunk_overlap e comparar Recall@k/MRR.
3. Adicionar reranker para reordenar os top documentos recuperados.